# Target Location Uncertainty
This tutorial demonstrates how to model uncertainty in target locations when imaging ground targets using a simple single-satellite BSK-RL environment. It also shows how target location uncertainty can grow over time when a target is not imaged.

## Loading Modules

In [ ]:
import numpy as np
from typing import Optional, Iterable, Any

from Basilisk.architecture import bskLogging
from Basilisk.utilities import orbitalMotion, macros
from bsk_rl import act, obs, sats
from bsk_rl.sim import dyn, fsw, world
from bsk_rl.sim.fsw import action
from bsk_rl.scene.targets import UniformTargets
from bsk_rl.data.unique_image_data import UniqueImageData, UniqueImageReward
from bsk_rl.utils.orbital import random_orbit
from bsk_rl.gym import SatelliteTasking

bskLogging.setDefaultLogLevel(bskLogging.BSK_WARNING)

## Making a Scenario with Location Uncertain Targets

To account for location uncertainty in the simulation process, the following parameters are assigned to each target using UniformTargets as a base:

* `search_area` represents a square region on the surface within which the estimated location is the center of the area.

* `acquiring_speed` represents the apparent ground-track speed, sampled uniformly to reflect orbital geometry and latitude effects.

* `scan_width` represents the fixed width of the imaging swath of the satellite.

* `search_duration` represents the total imaging duration, proportional to the number of sensor swaths required to cover the area.

* `image_time` represents the true imaging time within the duration window, drawn from a Gaussian distribution centered at the mid-point of $T_i$ ($\mu_i$) with standard deviation $0.3T_i$ ($\sigma_i$), truncated to nonnegative values. Approximately \(4.8\%\) of samples occur after the window ends.

* `speed` represents the target's ground speed, sampled uniformly to reflect typical ship speeds and fixed for each target.

In [ ]:
class AreaTargets(UniformTargets):

    def regenerate_targets(self) -> None:
        super().regenerate_targets()
        for target in self.targets:
            side_length = np.random.uniform(10.0, 100.0) * 1e3  # m
            target.search_area = side_length**2  # m^2
            target.acquiring_speed = np.random.uniform(3.0, 6.0) * 1e3  # m/s
            scan_width = 20.0 * 1e3  # m
            search_duration = (
                side_length * np.ceil(side_length / scan_width) / target.acquiring_speed
            )
            target.search_duration = search_duration  # s

            mean_duration = search_duration / 2.0
            std_duration = search_duration * 0.3
            image_time = np.random.normal(loc=mean_duration, scale=std_duration)
            image_time = max(0.0, image_time)
            target.image_time = image_time  # s

            target.speed = np.random.uniform(22e3 / 3600, 46e3 / 3600)  # m/s

## Making a Rewarder Considering Target Uncertainty

When targets have location uncertainty, imaging the highest-priority target is not always the optimal choice. If a target is not imaged for some time, its uncertainty grows, which can reduce future imaging effectiveness. Therefore, the reward function should balance target priority and uncertainty. One way to achieve this is by multiplying the priority by the uncertainty for each target.
The `UncertainMultiplyReward` is built on the [UniqueImageReward](../api_reference/data/index.rst) base class and defines the reward function as

$$
R(s, a, s') = 
\begin{cases}
p_n \cdot u_n, & \text{if } a = a_{\text{image}, n} \text{ and } t_{\text{search}} = t_n^{\ast} \\
0, & \text{otherwise}
\end{cases}
$$

where $p_n$ is the target priority, $u_n$ is the target uncertainty (defined as the search area normalized by the maximum initial search area of 100~km $\times$ 100~km), $t_{\text{search}}$ is the time spent searching the target, $t_n^{\ast}$ is the true imaging duration, and $a_{\text{image}, n}$ denotes the imaging action for the selected target $n$.
Unlike `UniqueImageReward`, this class removes the imaging list filter, allowing the same target to be imaged multiple times.

In [ ]:
from typing import TYPE_CHECKING

if TYPE_CHECKING:  # pragma: no cover
    from bsk_rl.sats import Satellite


class UncertainMultiplyReward(UniqueImageReward):

    def create_data_store(self, satellite: "Satellite") -> None:
        """Create a data store for a satellite.

        Args:
            satellite: Satellite to create a data store for.
        """
        satellite.data_store = self.data_store_type(
            satellite,
            initial_data=self.initial_data(satellite),
            **self.data_store_kwargs,
        )
        self.cum_reward[satellite.name] = 0.0

    def calculate_reward(
        self, new_data_dict: dict[str, UniqueImageData]
    ) -> dict[str, float]:
        """Reward images of targets based on their priority and current uncertainty.

        Targets may be imaged multiple times; each image earns reward based on
        ``self.reward_fn(target.priority * uncertainty)``, with the reward
        shared among all simultaneous images of the same target.

        Args:
            new_data_dict: Record of new images for each satellite

        Returns:
            reward: Cumulative reward across satellites for one step
        """
        reward = {}
        imaged_counts = {}
        for new_data in new_data_dict.values():
            for target in new_data.imaged:
                if target not in imaged_counts:
                    imaged_counts[target] = 0
                imaged_counts[target] += 1

        for sat_id, new_data in new_data_dict.items():
            reward[sat_id] = 0.0
            for target in new_data.imaged:
                uncertainty = target.search_area / ((100.0 * 1e3) ** 2)
                reward[sat_id] += (
                    self.reward_fn(target.priority * uncertainty)
                    / imaged_counts[target]
                )
        return reward

## Configuring the Satellite to Have Access to Target Uncertainty Information

The satellite has observations and actions associated with it that are relevant to the decision-making process. The observation space can be modified to include information about the targets and the uncertainty which allows better informed decision-making.

* [Observations](../api_reference/obs/index.rst): 
    - SatProperties: Body angular velocity, instrument pointing direction, body position, and body velocity in [dynamics model](../api_reference/sim/dyn/index.rst).
    - OpportunityProperties: Target priority, pointing error, pointing rate error, time until the target window opens, time until the target window closes, and uncertainty (normalized search area) for the next 32 targets.
    - Density: Upcoming target request density.
* [Actions](../api_reference/act/index.rst):
    - Image: Image target from upcoming 32 targets
* [Dynamics model](../api_reference/sim/dyn/index.rst): FullFeaturedDynModel is used.
* [Flight software model](../api_reference/sim/fsw/index.rst): SteeringImagerFSWModel used as the base model and modified for duration-based imaging.

In [ ]:
if TYPE_CHECKING:  # pragma: no cover
    from bsk_rl.scene.targets import (
        Target,
    )


class Density(obs.Observation):
    def __init__(
        self,
        interval_duration=60 * 3,
        intervals=10,
        norm=3,
    ):
        self.satellite: "sats.ImagingSatellite"
        super().__init__()
        self.interval_duration = interval_duration
        self.intervals = intervals
        self.norm = norm

    def get_obs(self):
        if self.intervals == 0:
            return []

        self.satellite.calculate_additional_windows(
            self.simulator.sim_time
            + (self.intervals + 1) * self.interval_duration
            - self.satellite.window_calculation_time
        )
        soonest = self.satellite.upcoming_opportunities_dict(types="target")
        rewards = np.array([opportunity.priority for opportunity in soonest])
        times = np.array([opportunities[0][1] for opportunities in soonest.values()])
        time_bins = np.floor((times - self.simulator.sim_time) / self.interval_duration)
        densities = [sum(rewards[time_bins == i]) for i in range(self.intervals)]
        return np.array(densities) / self.norm


class CustomFswModel(fsw.SteeringImagerFSWModel):
    @action
    def action_image(
        self,
        r_LP_P: Iterable[float],
        data_name: str,
        acquisitionTime: float,
        allowedTime: float,
    ) -> None:
        """Attempt to image a target at a location.

        This action sets the target attitude to one tracking a ground location. If the
        target is within the imaging constraints for the specified time, an image will be taken and stored in
        the data buffer. The instrument power sink will be active as long as the task is
        enabled.

        Args:
            r_LP_P: [m] Planet-fixed planet relative target location.
            data_name: Data buffer to store image data to.
            acquisitionTime: [s] Time required to acquire the image.
            allowedTime: [s] Maximum time allowed to attempt to acquire the image.
        """
        self.insControl.controllerStatus = 1
        self.dynamics.instrumentPowerSink.powerStatus = 1
        self.dynamics.imagingTarget.r_LP_P_Init = r_LP_P
        self.dynamics.instrument.nodeDataName = data_name
        self.insControl.imaged = 0
        self.insControl.useDurationImaging = True
        self.insControl.acquisitionTime = macros.sec2nano(acquisitionTime)
        self.insControl.allowedTime = macros.sec2nano(allowedTime)
        self.simulator.enableTask(self.LocPointTask.name + self.satellite.name)


class CustomSatComposed(sats.ImagingSatellite):
    observation_spec = [
        obs.SatProperties(
            dict(prop="omega_BN_B", norm=0.03),
            dict(prop="c_hat_H"),
            dict(prop="r_BN_P", norm=orbitalMotion.REQ_EARTH * 1e3),
            dict(prop="v_BN_P", norm=7616.5),
        ),
        Density(intervals=20, norm=5),
        obs.OpportunityProperties(
            dict(prop="priority"),
            dict(prop="r_LB_H", norm=orbitalMotion.REQ_EARTH * 1e3),
            dict(prop="target_angle", norm=np.pi / 2),
            dict(prop="target_angle_rate", norm=0.03),
            dict(prop="opportunity_open", norm=300.0),
            dict(prop="opportunity_close", norm=300.0),
            dict(
                prop="search_area",
                fn=lambda sat, opp: opp["object"].search_area,
                norm=(100.0 * 1e3) ** 2,
            ),
            type="target",
            n_ahead_observe=32,
        ),
    ]

    action_spec = [
        act.Image(n_ahead_image=32),
    ]

    dyn_type = dyn.FullFeaturedDynModel
    fsw_type = CustomFswModel

    def task_target_for_imaging(
        self, target: "Target", max_duration: Optional[float] = None
    ):
        """Task the satellite to image a target.

        Args:
            target: Selected target
            max_duration: [s] Maximum duration to wait for imaging. If None, wait until
                the end of the target's access window.
        """
        msg = f"{target} tasked for imaging, image_time: {target.image_time:.1f}, duration: {target.search_duration:.1f}"
        self.logger.info(msg)
        self.fsw.action_image(
            target.r_LP_P, self.buffer_name, target.image_time, target.search_duration
        )  # times in seconds
        self.enable_target_window(target, max_duration=max_duration)
        self.draw_imaging_line(target)
        self.latest_target = target

When instantiating a satellite, these parameters can be overriden with a constant or 
rerandomized every time the environment is reset using the ``sat_args`` dictionary.

In [ ]:
dataStorageCapacity = (
    20 * 8e6 * 100
)  # Large storage to avoid filling up in three orbits
batteryStorageCapacity = (
    1000.0 * 3600 * 2
)  # Large storage to avoid battery running out in three orbits
ALTITUDE = 800
T_ORBIT = (  # Orbital period
    2
    * np.pi
    * np.sqrt((orbitalMotion.REQ_EARTH + ALTITUDE) ** 3 / orbitalMotion.MU_EARTH)
)
sat_args = CustomSatComposed.default_sat_args(
    oe=lambda: random_orbit(
        alt=ALTITUDE,  # 800 km altitude
        i=45,  # 45 degrees inclination
    ),
    imageAttErrorRequirement=0.01,
    imageRateErrorRequirement=0.01,
    batteryStorageCapacity=batteryStorageCapacity,
    storedCharge_Init=lambda: np.random.uniform(0.4, 1.0) * batteryStorageCapacity,
    u_max=0.4,
    K1=0.25,
    K3=3.0,
    servo_P=150 / 5,
    nHat_B=np.array([0, 0, -1]),
    omega_max=np.radians(5.0),  # Maximum rate command in degrees per second
    imageTargetMinimumElevation=np.arctan(800 / 500),  # 58 degrees minimum elevation
    rwBasePower=20,
    maxWheelSpeed=1500,
    storageInit=lambda: np.random.randint(
        0 * dataStorageCapacity,
        0.01 * dataStorageCapacity,
    ),  # Initialize storage use close to zero
    wheelSpeeds=lambda: np.random.uniform(
        -1, 1, 3
    ),  # Initialize reaction wheel speeds close to zero
    dataStorageCapacity=dataStorageCapacity,
)

## Making Target Uncertainty Growing Environment

In order to model the growth of target location uncertainty over time, the step function of the gym environment is modified with the [SatelliteTasking](../api_reference/index.rst) base class. The target uncertainty grows based on each time step and the target's speed which is set in the `AreaTargets` class. 

In [ ]:
class SatelliteUncertaintyTargets(SatelliteTasking):
    def step(self, action) -> tuple[Any, float, bool, bool, dict[str, Any]]:
        """Take a single-satellite action, run parent step, and update target uncertainties."""
        target_selected = self.satellites[0].parse_target_selection(action)
        obs, reward, terminated, truncated, info = super().step(action)
        d_ts = info["d_ts"]

        targets = self.satellites[0].data_store.data.known
        area_factor = 20e3  # scan width in meters (20 km)

        # Compute all new values in vectorized form
        search_areas = np.array([t.search_area for t in targets])
        side_lengths = np.sqrt(search_areas)
        acquiring_speeds = np.array([t.acquiring_speed for t in targets])
        speeds = np.array([t.speed for t in targets])

        new_side_lengths = side_lengths + speeds * d_ts
        new_search_areas = new_side_lengths**2
        new_search_durations = (
            new_side_lengths
            * np.ceil(new_side_lengths / area_factor)
            / acquiring_speeds
        )

        means = new_search_durations / 2.0
        stds = new_search_durations * 0.3
        new_image_times = np.maximum(0.0, np.random.normal(loc=means, scale=stds))

        # Update the target information
        for t, area, dur, time in zip(
            targets, new_search_areas, new_search_durations, new_image_times
        ):
            t.search_area = area
            t.search_duration = dur
            t.image_time = time

        # Reset the imaged target's uncertainty
        if reward > 0.0:
            target_selected.search_area = 0.0
            target_selected.search_duration = 0.0
            target_selected.image_time = 0.0

        return obs, reward, terminated, truncated, info

## Initializing and Interacting with the Environment
For this example, the custom environment created above is used along with passing the satellite that is configured, the environment takes a [scenario](../api_reference/scene/index.rst), which defines the environment the satellite is acting in, and a [rewarder](../api_reference/data/index.rst), which defines how data collected from the scenario is rewarded.

In [ ]:
satellite = CustomSatComposed("EO", sat_args)
n_targets = (1000, 10000)

env = SatelliteUncertaintyTargets(
    satellite=satellite,
    world_type=world.GroundStationWorldModel,
    world_args=world.GroundStationWorldModel.default_world_args(),
    scenario=AreaTargets(n_targets=n_targets),
    rewarder=UncertainMultiplyReward(),
    sim_rate=0.5,
    max_step_duration=T_ORBIT,  # Duration long enough to prevent interruption of the imaging action
    time_limit=T_ORBIT * 3,  # three orbits
    log_level="INFO",
    failure_penalty=0.0,
    # disable_env_checker=True,  # For debugging
)

First, reset the environment. It is possible to specify the seed when resetting the environment.

In [ ]:
observation, info = env.reset(seed=1)

It is possible to print out the actions and observations. The composed satellite [action_description](../api_reference/sats/index.rst) returns a human-readable action map. Each satellite has the same action space and similar observation space.

In [ ]:
print("Actions:", env.satellites[0].action_description, "\n")
print("States:", env.unwrapped.satellites[0].observation_description, "\n")

# Using the composed satellite features also provides a human-readable state:
for satellite in env.unwrapped.satellites:
    for k, v in satellite.observation_builder.obs_dict().items():
        print(f"{k}:  {v}")

Then, run the simulation until timeout.

In [ ]:
while True:
    action_step = np.random.randint(0, 32)  # Random action

    observation, reward, terminated, truncated, info = env.step(action_step)

    if terminated or truncated:
        print("Episode complete.")
        break

After running the simulation, we can check the total reward, the number of unique targets that were imaged at least once, and the number of duplicated (re-imaged) targets collected by the rewarder.

In [ ]:
print("Total reward:", env.unwrapped.rewarder.cum_reward)
print(
    "Number of unique imaged targets:",
    len(set(env.unwrapped.rewarder.data.imaged)),
)
print(
    "Number of duplicated targets:",
    env.unwrapped.rewarder.data.duplicates,
)

Check [Training with RLlib PPO](../examples/rllib_training.ipynb) for an example on how to train the agent in this environment.